## 生成数据集

与 :numref:`sec_linear_scratch`中类似，我们首先[**生成数据集**]。

In [12]:
import numpy as np
import torch
from torch.utils import data
from d2l import torch as d2l

In [13]:
true_w = torch.tensor([2,-3.4])
true_b=4.2
features , labels = d2l.synthetic_data(true_w,true_b,1000)

## 读取数据集

我们可以[**调用框架中现有的API来读取数据**]。
我们将`features`和`labels`作为API的参数传递，并通过数据迭代器指定`batch_size`。
此外，布尔值`is_train`表示是否希望数据迭代器对象在每个迭代周期内打乱数据。


In [14]:
def load_array(data_arrays, batch_size, is_train=True):
    """Construct a PyTorch data iterator.
    
    Args:
        data_arrays: 包含特征和标签的元组或列表，如 (features, labels)
        batch_size: 每个批次的样本数量
        is_train: 是否为训练数据，True时打乱数据，False时保持顺序
    
    Returns:
        data.DataLoader: PyTorch数据加载器，用于迭代批次数据
    
    功能说明：
        1. 根据data_arrays创建TensorDataset对象，将所有数组打包为数据集
        2. 使用DataLoader封装数据集，按照batch_size分批
        3. 如果is_train=True，每个epoch会自动打乱数据顺序
        4. 返回可迭代的DataLoader对象，用于模型训练
    """
    # 这里的 * 是Python中的解包操作符（unpacking operator）。
    # 它会将传入的元组或列表（如 data_arrays）中的元素解包为独立的参数传递给 TensorDataset。
    dataset = data.TensorDataset(*data_arrays)
    return data.DataLoader(dataset, batch_size, shuffle=is_train)

batch_size = 10
# 这行代码调用我们上面定义的 load_array 函数来创建一个PyTorch数据迭代器 (DataLoader)。
# 它将特征 (features) 和标签 (labels) 作为元组传入，并指定了批量大小 (batch_size)。
# 返回的 data_iter 是一个可迭代对象，在模型训练时，可以用它每次取出一小批 (batch_size个) 乱序后的数据。
data_iter = load_array((features, labels), batch_size)

# iter(data_iter) 将 data_iter 对象转换为一个 Python 的迭代器。
# next(...) 函数则从这个迭代器中获取它的第一个元素。
# 在这里，它会返回数据迭代器中的第一个批次（batch），也就是包含 10 个特征样本和对应标签的一个元组或列表。
next(iter(data_iter))

[tensor([[-0.7277,  0.9300],
         [-0.2733, -1.6207],
         [ 1.3717, -1.4102],
         [-0.0607,  0.9304],
         [-0.1873, -0.3735],
         [ 0.7748, -1.1201],
         [ 1.3837, -0.9409],
         [-0.3795, -2.3741],
         [-0.8344,  0.3410],
         [-0.8306,  0.6341]]),
 tensor([[-0.4012],
         [ 9.1633],
         [11.7357],
         [ 0.9238],
         [ 5.0902],
         [ 9.5594],
         [10.1774],
         [11.5222],
         [ 1.3643],
         [ 0.3959]])]

使用框架的预定义好的层


In [15]:
from torch import nn
#nn 是neural network的缩写，包含了构建神经网络的各种模块和函数。
net = nn.Sequential(nn.Linear(2,1))
# nn.Sequential 是一个有序的容器，网络层将按照传入的顺序依次被添加到计算图中。
# nn.Linear(2, 1) 定义了一个线性变换层（即全连接层），其中 2 代表输入特征的维度个数，1 代表输出的维度个数。
# 综合起来，这行代码构建了一个用于线性回归的单层神经网络模型。

初始化模型参数

In [16]:
net[0].weight.data.normal_(0, 0.01)# 这行代码的作用是初始化神经网络中第一个层（即线性层）的权重参数。
# net[0] 访问神经网络中的第一个层（线性层）。
net[0].bias.data.fill_(0)# 这行代码的作用是初始化神经网络中第一个层（即线性层）的偏置参数。
# net[0] 访问神经网络中的第一个层（线性层）。

tensor([0.])

[**计算均方误差使用的是`MSELoss`类，也称为平方$L_2$范数**]。
默认情况下，它返回所有样本损失的平均值。


In [17]:
loss = nn.MSELoss()


实例化sgd实例

In [18]:
trainer = torch.optim.SGD(net.parameters(),lr=0.03)
# 这行代码实例化了一个随机梯度下降（SGD）优化器。
# net.parameters() 获取神经网络中所有可学习的参数（即权重和偏置），并将它们交由优化器管理。
# lr=0.03 指定了优化过程中的学习率（learning rate）为0.03，它决定了每次参数更新的步长大小。

## 训练

通过深度学习框架的高级API来实现我们的模型只需要相对较少的代码。
我们不必单独分配参数、不必定义我们的损失函数，也不必手动实现小批量随机梯度下降。
当我们需要更复杂的模型时，高级API的优势将大大增加。
当我们有了所有的基本组件，[**训练过程代码与我们从零开始实现时所做的非常相似**]。

回顾一下：在每个迭代周期里，我们将完整遍历一次数据集（`train_data`），
不停地从中获取一个小批量的输入和相应的标签。
对于每一个小批量，我们会进行以下步骤:

* 通过调用`net(X)`生成预测并计算损失`l`（前向传播）。
* 通过进行反向传播来计算梯度。
* 通过调用优化器来更新模型参数。

为了更好的衡量训练效果，我们计算每个迭代周期后的损失，并打印它来监控训练过程。


In [19]:
"""
=====================================
【训练循环结构图】
[开始每个 epoch (轮次)]
      │
      ▼
[取出一小批数据 X, y] <────────┐
      │                        │
      ▼                        │
[1. 计算损失(前向传播)]        │
    l = loss(net(X), y)        │
      │                        │
      ▼                        │
[2. 梯度清零(准备算梯度)]      │ (循环直到这一个 epoch 的所有批次都取完)
    trainer.zero_grad()        │
      │                        │
      ▼                        │
[3. 计算梯度(反向传播)]        │
    l.backward()               │
      │                        │
      ▼                        │
[4. 参数更新(走一步)]          │
    trainer.step()  ───────────┘
      │
      ▼
[一个 epoch 结束，计算整个数据集上的 Loss 并评估打印]
=====================================
"""
num_epochs = 3 # 训练的总轮数（将整个数据集遍历3次）

for epoch in range(num_epochs): # 开始外层循环，每次循环代表过完一整遍数据
    for X, y in data_iter: # 内层循环：从数据迭代器 data_iter 中按批次取出特征 X 和标签 y
        l = loss(net(X), y) # 前向传播：用模型 net 预测 X 的结果，然后算出与真实标签 y 的误差损失
        trainer.zero_grad() # 梯度清零：把上一次计算留下的梯度清零（PyTorch会默认累加往期的梯度，所以每次得清空）
        l.backward() # 反向传播：根据损失 l，自动反推计算出每一个模型参数应该调整的方向和大小（即“梯度”）
        trainer.step() # 参数更新：调用优化器 trainer (也就是SGD)，让模型参数顺着刚算出来的梯度去更新自己
        
    # 这个内层循环结束代表所有数据都被学习过一遍了。
    # 接着拿全部的数据特征(features)和标签(labels)再算一次损失，看看模型现在的学习成果
    l = loss(net(features), labels) 
    print(f'epoch {epoch + 1}, loss {l:f}') # 打印当前是第几轮，以及此时计算出的全量损失


epoch 1, loss 0.000239
epoch 2, loss 0.000099
epoch 3, loss 0.000099
